In [12]:
!pip install spacy

Defaulting to user installation because normal site-packages is not writeable
  Using cached spacy-3.8.18-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.16-cp312-cp312-win_amd64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.16-cp312-cp312-win_amd64.whl.metadata (10.0 kB)
  Using cached preshed-3.0.15-cp312-cp312-win_amd64.whl.metadata (5.4 kB)
  Using cached thinc-8.3.14-cp312-cp312-win_amd64.whl.metadata (15 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.6-cp312-cp312-win_amd64.whl.metadata (20 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached confection-1.3.4-py3-none-any.whl.metadata (19 kB)
  Using cached typer-0.29.1-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.5

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
reflex 0.7.11 requires packaging<26,>=24.2, but you have packaging 24.1 which is incompatible.
reflex 0.7.11 requires typing-extensions>=4.13.0, but you have typing-extensions 4.12.2 which is incompatible.
reflex-hosting-cli 0.1.49 requires packaging>=24.2, but you have packaging 24.1 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\debia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [13]:
!python -m spacy download es_core_news_sm

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     ----------- ---------------------------- 3.7/12.9 MB 19.8 MB/s eta 0:00:01
     --------------------------------------  12.8/12.9 MB 38.4 MB/s eta 0:00:01
     --------------------------------------- 12.9/12.9 MB 31.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\debia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
import spacy
# Cargar modelo ahora en español 
try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    print("Descargando modelo...")
    from spacy.cli import download
    download("es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")

# TEXTO DE ENTRADA 
with open("harry_potter.txt", "r", encoding="utf-8") as f:
    texto_harry = f.read()

# Ahora 'texto_harry' contiene todo el texto del archivo
print(f"Texto cargado con éxito. Longitud: {len(texto_harry)} caracteres.")

# 1. TOKENIZACIÓN
# SpaCy procesa el texto y crea el objeto 'doc' lleno de metadatos
doc = nlp(texto_harry)

# Mostrar los primeros 15 tokens para entender cómo "ve" la máquina el texto
print(f"--- 1. Tokenización (Total tokens: {len(doc)}) ---")
print([token.text for token in doc][:20]) 

Texto cargado con éxito. Longitud: 1529 caracteres.
--- 1. Tokenización (Total tokens: 321) ---
['Sus', 'ojos', 'se', 'clavaron', 'en', 'el', 'hogar', 'apagado', ',', 'con', 'las', 'bolsas', 'de', 'patatas', 'fritas', 'arrugadas', ',', 'y', 'dejó', 'escapar']


In [15]:
#Eliminacion de Ruido

# 2. FILTRADO DE STOP WORDS
# Separamos lo que aporta valor semántico del "pegamento" gramatical

tokens_relevantes = []
tokens_ruido = []

for token in doc:
    # Filtramos si es stopword o si es puntuación
    if not token.is_stop and not token.is_punct and token.text.strip():
        tokens_relevantes.append(token.text)
    elif token.is_stop:
        tokens_ruido.append(token.text)

print(f"\n--- 2. Filtrado de Stop Words ---")
print(f"Palabras eliminadas (Ruido): {tokens_ruido[:10]}...")
print(f"Palabras conservadas (Contenido): {tokens_relevantes[:10]}...")
print(f"Reducción de tamaño: de {len(doc)} a {len(tokens_relevantes)} tokens.")


--- 2. Filtrado de Stop Words ---
Palabras eliminadas (Ruido): ['Sus', 'se', 'en', 'el', 'con', 'las', 'de', 'y', 'dejó', 'una']...
Palabras conservadas (Contenido): ['ojos', 'clavaron', 'hogar', 'apagado', 'bolsas', 'patatas', 'fritas', 'arrugadas', 'escapar', 'risa']...
Reducción de tamaño: de 321 a 117 tokens.


In [16]:
# 3. LEMATIZACIÓN Y NORMALIZACIÓN FINAL
# Reducimos las palabras a su raíz (Lema) y estandarizamos a minúsculas
# Objetivo: Que "hablo", "hablaré" y "habla" cuenten como el mismo concepto: "hablar"

tokens_normalizados = []
cambios_interesantes = []

for token in doc:
    # Aplicamos los mismos filtros de calidad que en el paso 2
    if not token.is_stop and not token.is_punct and token.text.strip():
        
        # AQUÍ OCURRE LA MAGIA: 
        # 1. Extraemos el lema (token.lemma_)
        # 2. Convertimos a minúsculas (.lower())
        lema = token.lemma_.lower()
        tokens_normalizados.append(lema)
        
        # Para fines educativos: Guardamos casos donde la palabra cambió drásticamente
        # Ej: "fui" -> "ir"
        if token.text.lower() != lema:
            cambios_interesantes.append(f"{token.text} ➡ {lema}")

print(f"\n--- 3. Lematización y Normalización ---")
print(f"Total de tokens procesados: {len(tokens_normalizados)}")
print(f"Ejemplos de transformaciones (Palabra original ➡ Lema):")
# Mostramos solo los primeros 5 cambios para no saturar la pantalla
print(cambios_interesantes[:10]) 

print(f"\nResultado final (Primeros 10 tokens):")
print(tokens_normalizados[:10])


--- 3. Lematización y Normalización ---
Total de tokens procesados: 117
Ejemplos de transformaciones (Palabra original ➡ Lema):
['ojos ➡ ojo', 'clavaron ➡ clavar', 'bolsas ➡ bolsa', 'patatas ➡ patata', 'fritas ➡ frito', 'arrugadas ➡ arrugado', 'despectiva ➡ despectivo', 'inclinó ➡ inclinar', 'podían ➡ poder', 'inundó ➡ inundar']

Resultado final (Primeros 10 tokens):
['ojo', 'clavar', 'hogar', 'apagado', 'bolsa', 'patata', 'frito', 'arrugado', 'escapar', 'risa']


In [17]:
 #Stemming. 
!pip install nltk

Defaulting to user installation because normal site-packages is not writeable
  Using cached nltk-3.10.4-py3-none-any.whl.metadata (5.0 kB)
  Using cached regex-2026.11.15-cp312-cp312-win_amd64.whl.metadata (41 kB)
Using cached nltk-3.10.4-py3-none-any.whl (1.7 MB)
Using cached regex-2026.11.15-cp312-cp312-win_amd64.whl (281 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\debia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [18]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\debia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [19]:
import pandas as pd
from nltk.stem import SnowballStemmer

# Configuración del Stemmer (NLTK)
stemmer = SnowballStemmer("spanish")
# Comparativa lado a lado
data_comparativa = []

for token in doc:
    # Solo analizamos palabras, ignoramos puntuación y espacios para claridad
    if not token.is_punct and not token.is_space:
        
        # A. STEMMING (Corte de sufijos)
        raiz_stem = stemmer.stem(token.text)
        
        # B. LEMATIZACIÓN (Análisis morfológico de SpaCy)
        # Nota: SpaCy usa el contexto para saber si 'fui' es 'ir' o 'ser'
        lema = token.lemma_
        
        data_comparativa.append({
            "Original": token.text,
            "Stemming (Corte)": raiz_stem,
            "Lematización (Diccionario)": lema,
            "¿Coinciden?": raiz_stem == lema
        })

# Crear DataFrame para visualizar
df = pd.DataFrame(data_comparativa)

print(f"\n--- 3. Stemming vs Lematización ---")
# Mostramos palabras interesantes donde se vea la diferencia
# Buscamos verbos conjugados o plurales
palabras_interesantes = ["hombres", "olvidado", "eres", "domesticado", "invisible", "ojos"]
filtro = df[df["Original"].isin(palabras_interesantes)]

print(filtro.to_string(index=False))

print("\n--- Visualización completa de los primeros 10 tokens ---")
print(df.head(10).to_string(index=False))


--- 3. Stemming vs Lematización ---
Original Stemming (Corte) Lematización (Diccionario)  ¿Coinciden?
    ojos             ojos                        ojo        False
    ojos             ojos                        ojo        False

--- Visualización completa de los primeros 10 tokens ---
Original Stemming (Corte) Lematización (Diccionario)  ¿Coinciden?
     Sus              sus                         su        False
    ojos             ojos                        ojo        False
      se               se                         él        False
clavaron             clav                     clavar        False
      en               en                         en         True
      el               el                         el         True
   hogar              hog                      hogar        False
 apagado             apag                    apagado        False
     con              con                        con         True
     las              las                      